# BharatCRS V7 - Transformers Multi-Task Training
This notebook trains the 6-domain, 20-issue civic classifier using Muril.

**Instructions:**
1. Upload `bharatcrs_v7_clean.csv` to this Colab session.
2. Upload `label_maps_v7.json` if you want to keep label indices consistent.
3. Run all cells.

In [ ]:
!pip install transformers datasets torch onnx onnxruntime

In [ ]:
import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModel, AutoConfig
from torch import nn
from torch.utils.data import DataLoader, Dataset

# --- CONFIG ---
MODEL_NAME = 'google/muril-base-cased'
MAX_LEN = 256
BATCH_SIZE = 8
EPOCHS = 15
LR = 1e-5

class MultiTaskCivicClassifier(nn.Module):
    def __init__(self, n_domains, n_issues):
        super().__init__()
        self.bert = AutoModel.from_pretrained(MODEL_NAME)
        hidden_size = self.bert.config.hidden_size
        
        self.domain_head = nn.Linear(hidden_size, n_domains)
        self.issue_head = nn.Linear(hidden_size, n_issues)
        self.severity_head = nn.Sequential(nn.Linear(hidden_size, 1), nn.Sigmoid())
        self.safety_head = nn.Linear(hidden_size, 1)
        self.vuln_head = nn.Linear(hidden_size, 1)
        
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.pooler_output
        
        return (
            self.domain_head(pooled),
            self.issue_head(pooled),
            self.severity_head(pooled),
            self.safety_head(pooled),
            self.vuln_head(pooled)
        )

print("Model Architecture defined for 5 output heads.")

### Data Loading & Export Logic
After training, we export to ONNX for use in the BharatCRS backend.